# Import

In [ ]:
%load_ext autoreload
%autoreload 2

import traceback    
import os ; import sys
sys.path.insert(0, os.path.abspath(os.path.join('./lib')))

import test
import utilities
import harmonics
import detect_ignition

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 

from typing import Dict, List, Optional, Sequence, Tuple

from scipy.signal import firwin, filtfilt, hilbert, welch
from dataclasses import dataclass
from dataclasses import asdict

from scipy.signal import savgol_filter


import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 128

# Load EEG Data

In [ ]:
FILES = [
    'data/test schumann_EPOCX_111270_2023.04.23T14.50.35.05.00.md.pm.bp.csv',
    'data/Test_06.11.20_14.28.18.md.pm.bp.csv',
    'data/20201229_29.12.20_11.27.57.md.pm.bp.csv',
    'data/med_EPOCX_111270_2021.06.12T09.50.52.04.00.md.bp.csv',
    'data/binaural_EPOCX_111270_2021.06.17T10.04.52.04.00.md.bp.csv',
]

def list_csv_files(directory):
    return [(directory+"/"+f) for f in os.listdir(directory) if f.endswith('.csv')]
    
KAGGLE = list_csv_files("data/mainData")
MPENG = list_csv_files("data/mpeng")
MPENG1 = list_csv_files("data/mpeng1")
MPENG2 = list_csv_files("data/mpeng2")
VEP = list_csv_files("data/vep")
PHYSF = list_csv_files("data/PhySF")


ELECTRODES = ['EEG.AF3','EEG.AF4',
              'EEG.F7','EEG.F8',
              'EEG.F3','EEG.F4',
              'EEG.FC5','EEG.FC6',
              'EEG.P7','EEG.P8',
              'EEG.O1','EEG.O2',
              'EEG.T7','EEG.T8']

FILENAME = FILES[0]

RECORDS = utilities.load_eeg_csv(FILENAME, electrodes=ELECTRODES)

# Estimate SR Harmonics

In [ ]:
CANON = [7.8, 14, 20, 26] #, 32.5, 39.0, 45.0, 53.3]

HARMONICS = harmonics.estimate_session_sr_harmonics(RECORDS, ELECTRODES, 128, canonical_harmonics=CANON, search_band=1)

print("SR estimate: ",["{:.2f}".format(x) for x in HARMONICS])

# Detect Ignitions & Six Panel

In [ ]:
SESSION_NAME = os.path.splitext(os.path.basename(FILENAME))[0]
print("######################### PROCESSING: ", SESSION_NAME)

IGN_OUT, IGNITION_WINDOWS = detect_ignition.detect_ignitions_session(
    RECORDS,
    sr_channel= "EEG.F4",
    eeg_channels= ELECTRODES,
    time_col= "timestamp",
    out_dir= 'exports_ignitions/S01',
    center_hz= HARMONICS[0], 
    half_bw_hz= 1,
    smooth_sec= 0.1,
    z_thresh= 3,
    min_isi_sec= 2.0, 
    window_sec= 20, 
    merge_gap_sec= 5.0,
    R_band= (HARMONICS[0]-1.0,HARMONICS[0]+1.0),
    R_win_sec= 1.0, 
    R_step_sec= 0.25,
    eta_pre_sec= 3.0, 
    eta_post_sec= 3.0,
    sr_reference= 'auto-SSD', 
    seed_method= 'latency',
    pel_band= (35,58), 
    harmonics= None, # (2,3,4,5,6,7),
    harmonics_hz= HARMONICS,
    harmonic_bw_hz =None, # : Optional[float]
    make_passport= True,
    show= True,
    verbose = True,
    session_name = SESSION_NAME
)


for idx, IGN_WIN in enumerate(IGNITION_WINDOWS):
    
    CFG = test.FeaturePackCfg(
        channels=ELECTRODES, time_col='Timestamp', fs=128,
        win_sec=6, 
        step_sec=0.15,
        sr_centers=IGN_OUT['events']['ignition_freqs'][idx][:3],
        ladder=IGN_OUT['events']['ignition_freqs'][idx],
        bw_hz=1,
    )
    test.six_panel(RECORDS,ELECTRODES,IGN_WIN,IGN_OUT,IGN_OUT['events']['ignition_freqs'][idx],CFG)

# Six Panel - Batch

In [ ]:
CANON = [7.8, 14, 20, 26]
HALF_BW = 0.8

RESULTS = pd.DataFrame()

for _file in PHYSF:
    
    # SESSION NAME
    _session_name = os.path.splitext(os.path.basename(_file))[0]
    print("Session: ",_session_name)

    # LOAD FILE
    _records = utilities.load_eeg_csv(_file, electrodes=ELECTRODES,header=0)

    # ESTIMATE SR HARMONICS
    _harmonics = harmonics.estimate_session_sr_harmonics(_records, ELECTRODES, 128, canonical_harmonics=CANON, search_band=0.5)
    print("SR estimate: ",["{:.2f}".format(x) for x in _harmonics])

    # DETECT IGNITIONS
    _ign_out, _ign_windows = detect_ignition.detect_ignitions_session(
        _records,
        sr_channel= "EEG.F4",
        eeg_channels= ELECTRODES,
        time_col= "timestamp",
        out_dir= 'exports_ignitions/S01',
        center_hz= _harmonics[0], 
        half_bw_hz= HALF_BW,
        smooth_sec= 0.25,
        z_thresh= 3,
        min_isi_sec= 2.0, 
        window_sec= 20, 
        merge_gap_sec= 5.0,
        R_band= (_harmonics[0]-HALF_BW,_harmonics[0]+HALF_BW),
        R_win_sec= 1.0, 
        R_step_sec= 0.25,
        eta_pre_sec= 3.0, 
        eta_post_sec= 3.0,
        sr_reference= 'auto-SSD', 
        seed_method= 'latency',
        pel_band= (35,58), 
        harmonics= None, # (2,3,4,5,6,7),
        harmonics_hz= _harmonics,
        harmonic_bw_hz= None, # : Optional[float]
        make_passport= False,
        show= True,
        verbose = True,
        session_name = _session_name
    )

    # STORE SESSION RESULTS
    RESULTS = pd.concat([RESULTS,_ign_out['events']],ignore_index=True)

    # IGNITION WINDOWS
    for idx, _ign_win in enumerate(_ign_windows):
        
        # FEATURE PACK CONFIG
        _cfg = test.FeaturePackCfg(
            channels=ELECTRODES, time_col='Timestamp', fs=128,
            win_sec=6, 
            step_sec=0.15,
            sr_centers=_ign_out['events']['ignition_freqs'][idx][:3],
            ladder=_ign_out['events']['ignition_freqs'][idx],
            bw_hz=HALF_BW,
        )

        # SIX PANEL
        test.six_panel(_records,ELECTRODES,_ign_win,_ign_out,_ign_out['events']['ignition_freqs'][idx],_cfg)

In [ ]:
print(len(RESULTS))

In [ ]:
RESULTS.to_csv("physf-3-sr-new.csv",index=False)